# A100 Training Controller
This notebook acts as the launcher for training the models on the NVIDIA A100 environment.
Run the cells sequentially to verify the environment, train the models, and view the final metrics.


In [ ]:
# 1. Python/PyTorch/CUDA environment check
import sys
import torch

print(f"Python Version: {sys.version}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA Device Name: {torch.cuda.get_device_name(0)}")


In [ ]:
# 2. nvidia-smi
!nvidia-smi


In [ ]:
# 3. Project directory check
import os
print(f"Current Directory: {os.getcwd()}")
print("Listing files in root:")
!ls -la


In [ ]:
# 4. Dataset existence check
dataset_path = "data/raw/dataset.csv"
if os.path.exists(dataset_path):
    print(f"✅ Dataset found at {dataset_path} ({os.path.getsize(dataset_path) / (1024*1024*1024):.2f} GB)")
else:
    print(f"❌ Dataset NOT found at {dataset_path}. Please place the CIC-IDS2017 dataset there before continuing.")


In [ ]:
# 5. Dependency / Environment information
!pip freeze | grep -E "torch|xgboost|scikit-learn|pandas|numpy"


In [ ]:
# 6. Run Classical Models
!PYTHONPATH=. python3 ml/training/train_models.py


In [ ]:
# 7. Run PyTorch Training (CUDA)
!PYTHONPATH=. python3 ml/training/train_dl.py


In [ ]:
# 8. Run Autoencoder Holdout Experiment
!PYTHONPATH=. python3 ml/training/train_anomaly.py --holdout-attack "DoS Hulk"


In [ ]:
# 9. Load and Display Final Metrics
import json
import pandas as pd

try:
    with open("models/experiment_log.json", "r") as f:
        logs = json.load(f)
        df_classical = pd.DataFrame([
            {
                "Model": log["model"], 
                "F1_Macro": log["metrics"]["f1_macro"],
                "ROC_AUC": log["metrics"].get("roc_auc", "N/A"),
                "Latency_ms": log["metrics"]["inference_latency_ms"]
            } for log in logs
        ])
    print("--- Classical Models ---")
    display(df_classical)
except FileNotFoundError:
    print("Classical logs not found.")
    
try:
    with open("models/dl_model/metadata.json", "r") as f:
        dl = json.load(f)
    print("\n--- PyTorch MLP ---")
    print(f"F1 Macro: {dl['metrics']['f1_macro']:.4f} | ROC_AUC: {dl['metrics'].get('roc_auc', 'N/A')}")
except FileNotFoundError:
    print("Deep Learning logs not found.")
    
try:
    with open("models/anomaly/anomaly_metadata.json", "r") as f:
        ae = json.load(f)
    print("\n--- Autoencoder (Zero-Day Detection) ---")
    print(f"Benign FPR: {ae.get('fpr_benign', 'N/A')}")
    print(f"Holdout Detection Rate: {ae.get('dr_holdout', 'N/A')}")
except FileNotFoundError:
    print("Anomaly Detection logs not found.")
